# One SPY put and one SPY call at two closes: spot, premium, delta, gamma, and the dealer's hedge

Theta Data end-of-day quotes and Greeks for two real SPY options (whole-chain daily snapshots cached by `gex_history_spy`).
First a put the dealer sold (negative dealer gamma) before a 2.6% fall, then a call the dealer bought (positive dealer gamma)
before a 10.5% rise. In both, spot reaches the strike, and the dealer trades shares to get back to delta neutral.

In [1]:
import datetime as dt
from pathlib import Path

import pandas as pd
import polars as pl


def example(day0, day1, exp, strike, right, dealer_position, contracts=1):
    """dealer_position: -1 the dealer sold the option, +1 the dealer bought it. Table and hedge for the two closes."""
    l0, l1 = f"{day0} close", f"{day1} close"
    rows = {}
    for label, day in ((l0, day0), (l1, day1)):
      d = pl.read_parquet(f"data/spy_greeks_{day}.parquet")
      d = d.filter((pl.col("right") == right) & (pl.col("strike") == strike) & (pl.col("expiration") == str(exp)))
      assert d.height == 1, f"{d.height} rows for the {strike:.0f} {right} expiring {exp} on {day}"
      r = d.to_pandas().iloc[0]
      rows[label] = {"days to expiry": (exp - day).days, "spot": r["underlying_price"], "premium (mid)": (r["bid"] + r["ask"]) / 2.0,
                       "implied volatility": r["implied_vol"], "delta": r["delta"], "gamma": r["gamma"],
                       "dealer gamma": dealer_position * r["gamma"]}
    table = pd.DataFrame(rows)
    table["change"] = table[l1] - table[l0]

    s0, s1 = table.loc["spot", l0], table.loc["spot", l1]
    d0, d1 = table.loc["delta", l0], table.loc["delta", l1]
    g0 = table.loc["gamma", l0]
    pos0, pos1 = dealer_position * 100 * contracts * d0, dealer_position * 100 * contracts * d1   # position delta in shares
    hedge0, hedge1 = -round(pos0), -round(pos1)                                                     # shares held for delta neutral (negative = short)
    name = right.lower()
    opened = f"{'buy' if dealer_position > 0 else 'sell'} {contracts} {name}, {'short' if hedge0 < 0 else 'buy'} {abs(hedge0)} shares"
    rehedge = f"{'buy' if hedge1 > hedge0 else 'sell'} {abs(hedge1 - hedge0)} shares"

    shown = table.round(4).astype(object)
    shown.loc["dealer action"] = [opened, rehedge, ""]
    print(f"SPY {strike:.0f} {name} expiring {exp}, dealer {'long' if dealer_position > 0 else 'short'} {contracts}")
    display(shown)
    print(f"{l0}: dealer {'buys' if dealer_position > 0 else 'sells'} {contracts} {name}, delta {d0:+.3f}. Position delta {pos0:+.1f} shares. "
          f"Dealer {'shorts' if hedge0 < 0 else 'buys'} {abs(hedge0)} shares. Book delta 0.")
    print(f"{l1}: spot {s0:.2f} to {s1:.2f} ({s1 - s0:+.2f}, {(s1 / s0 - 1) * 100:+.1f}%). Delta {d0:+.3f} to {d1:+.3f}. Position delta {pos1:+.1f} shares. "
          f"Hedge still {hedge0} shares. Book delta {pos1 + hedge0:+.1f} shares.")
    print(f"Re-hedge: {rehedge}. Hedge now {hedge1} shares. Book delta 0.")
    print(f"Gamma check: gamma at {l0} times the spot change = {g0:.4f} times {s1 - s0:+.2f} = {g0 * (s1 - s0):+.3f}; actual delta change {d1 - d0:+.3f} "
          f"(gamma rose as spot reached the strike, so the straight-line estimate undershoots).")


In [2]:
example(dt.date(2026, 6, 4), dt.date(2026, 6, 5), dt.date(2026, 6, 8), 737.0, "PUT", dealer_position=-1)

SPY 737 put expiring 2026-06-08, dealer short 1


,2026-06-04 close,2026-06-05 close,change
days to expiry,4.0,3.0,-1.0
spot,757.09,737.55,-19.54
premium (mid),0.125,5.33,5.205
implied volatility,0.1374,0.214,0.0766
delta,-0.0284,-0.4746,-0.4462
gamma,0.0059,0.0278,0.0219
dealer gamma,-0.0059,-0.0278,-0.0219
dealer action,"sell 1 put, short 3 shares",sell 44 shares,


2026-06-04 close: dealer sells 1 put, delta -0.028. Position delta +2.8 shares. Dealer shorts 3 shares. Book delta 0.
2026-06-05 close: spot 757.09 to 737.55 (-19.54, -2.6%). Delta -0.028 to -0.475. Position delta +47.5 shares. Hedge still -3 shares. Book delta +44.5 shares.
Re-hedge: sell 44 shares. Hedge now -47 shares. Book delta 0.
Gamma check: gamma at 2026-06-04 close times the spot change = 0.0059 times -19.54 = -0.115; actual delta change -0.446 (gamma rose as spot reached the strike, so the straight-line estimate undershoots).


In [3]:
# 2. the dealer bought a call; SPY rose 10.5% to the strike the next day
example(dt.date(2025, 4, 8), dt.date(2025, 4, 9), dt.date(2025, 4, 14), 550.0, "CALL", dealer_position=+1)

SPY 550 call expiring 2025-04-14, dealer long 1


,2025-04-08 close,2025-04-09 close,change
days to expiry,6.0,5.0,-1.0
spot,496.48,548.62,52.14
premium (mid),0.41,5.62,5.21
implied volatility,0.4384,0.239,-0.1994
delta,0.0375,0.4783,0.4408
gamma,0.0029,0.0259,0.023
dealer gamma,0.0029,0.0259,0.023
dealer action,"buy 1 call, short 4 shares",sell 44 shares,


2025-04-08 close: dealer buys 1 call, delta +0.037. Position delta +3.8 shares. Dealer shorts 4 shares. Book delta 0.
2025-04-09 close: spot 496.48 to 548.62 (+52.14, +10.5%). Delta +0.037 to +0.478. Position delta +47.8 shares. Hedge still -4 shares. Book delta +43.8 shares.
Re-hedge: sell 44 shares. Hedge now -48 shares. Book delta 0.
Gamma check: gamma at 2025-04-08 close times the spot change = 0.0029 times +52.14 = +0.151; actual delta change +0.441 (gamma rose as spot reached the strike, so the straight-line estimate undershoots).


Short put: spot fell to the strike, premium, the size of delta, gamma and implied volatility rose, dealer gamma became more
negative, and the dealer sold shares into the fall (with the move). Long call: spot rose to the strike, premium, delta and
gamma rose while implied volatility fell, dealer gamma became more positive, and the dealer sold shares into the rise
(against the move). Educational analysis, not a trading strategy.